In [1]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/CreditCardFraud'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
import os

import time
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.filterwarnings('ignore')

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics         import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics         import roc_auc_score
from sklearn.metrics         import precision_recall_curve, classification_report
from sklearn.datasets         import make_classification

# Model import
from sklearn.tree           import DecisionTreeClassifier
from sklearn.ensemble       import RandomForestClassifier
from sklearn.ensemble       import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.linear_model   import LinearRegression
from sklearn.svm            import SVC
from sklearn.metrics        import classification_report
from xgboost                import XGBClassifier
from xgboost                import plot_importance
from lightgbm               import LGBMClassifier
from catboost               import CatBoostClassifier

# hyperopt 용
from hyperopt               import hp

# 사용자 Functions import
import HyperParams          as HP 
import utils.data_sampling  as ds 

from utils import user_utils    as uu
from utils import preprocessing as pp
from utils import data_sampling as ds
from utils import model_utils   as mu
from utils import modeling      as mo

In [15]:
# 결과받을 딕셔너리
results = {}
team_rs = 23 # 우리팀 random_state

In [7]:
#1. 데이터 로딩
raw_df = pp.ccf_load_data()

데이터 로드 성공: (284807, 31)


In [8]:
# 2. Time 컬럼 삭제 , 데이터,타겟 분리
X_features, y_target = pp.split_features_target(raw_df, cols= 'Time')

In [9]:
# 3. 이상치를 경계값으로 치환
cap_X_feature = pp.cap_outliers(X_features)

In [10]:
# 4.1 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = pp.data_split(cap_X_feature, y_target)

In [11]:
# 4.2 Over Sampling 하는 경우
X_over, y_over = ds.oversampling_smote(X_train, y_train)

✅ SMOTE 오버샘플링 완료
   원본 샘플 수: 227845 (Class 0: 227451, Class 1: 394)
   샘플링 후: 454902 (Class 0: 227451, Class 1: 227451)


In [12]:
# 5.1 학습/검증 데이터 분리
X_tr, X_val, y_tr, y_val = pp.data_split(X_train, y_train, size=0.4)

In [13]:
# 5.2 Over Sampling한 경우 학습/검증 데이터 분리
X_tr, X_val, y_tr, y_val = pp.data_split(X_over, y_over, size=0.4)

In [ ]:
#6 하이퍼파라미터
tuner = uu.HyperOptTuner(max_evals=100, random_state=team_rs)

In [ ]:
# 모델별 스페이스 생성 : 예시는 catboost 
# learning_rate (0.01–0.2), max_depth (3–10), n_estimators (100–1000), subsample (0.5–1.0), colsample_bytree (0.5–1.0)    
xgb_search_space = {
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 50),
    'subsample': hp.quniform('subsample', 0.5, 1.0, 0.1),  
    'max_depth': hp.quniform('max_depth', 3, 10, 1), 
    'learning_rate': hp.loguniform('learning_rate', np.log(0.001), np.log(0.3)),  # 수정: loguniform이 더 적합
    'colsample_bytree': hp.quniform('colsample_bytree', 0.5, 1.0, 0.1),  
    'scale_pos_weight': hp.quniform('scale_pos_weight', 1, 100, 1),  
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),  
    'gamma': hp.uniform('gamma', 0, 5),
    'reg_alpha': hp.loguniform('reg_alpha', np.log(0.001), np.log(10)),  # 수정: 0 제외, 범위 확대
    'reg_lambda': hp.loguniform('reg_lambda', np.log(0.001), np.log(10)),  # 수정: loguniform 사용
}

# 모델 생성
xgb_clf = XGBClassifier()

# 파라미터 지정
best_params, best_xgm, trials, exec_time = tuner.tune(
    xgb_clf, X_tr, y_tr, X_val, y_val, xgb_search_space
)

# best모델로 결과출력
# 모델명 규칙 : 2~3자리 모델명 + _ho_best
model_name = 'xgb_ho_best2'
results[model_name] =uu.get_model_train_eval(
    best_xgm, model_name, X_train, X_test, y_train, y_test, best_params
)



XGBClassifier 튜닝 시작
100%|██████████| 100/100 [16:19<00:00,  9.79s/trial, best loss: -1.0]             

튜닝 시간: 979.35초
최적 recall: 1.0000

최적 모델의 전체 평가 점수:
- roc_auc: 0.9995
- f1: 0.9758
- precision: 0.9528
- recall: 1.0000
- accuracy: 0.9752

최적 하이퍼파라미터:
-colsample_bytree: 0.6000000000000001
-gamma: 4.787185438562684
-learning_rate: 0.005262189759984318
-max_depth: 5
-min_child_weight: 6
-n_estimators: 700
-reg_alpha: 0.020391918886446342
-reg_lambda: 0.0969098930319443
-scale_pos_weight: 14.0
-subsample: 0.9
-random_state: 23
✓ 모델 저장 완료: ../models\xgb_ho_best2.pkl
  파일 크기: 1.47 MB
folder = c:\big20\git\big20-ML-project2-team3\CreditCardFraud\results
{'AUC': 0.9731, '정확도': 0.9994, '정밀도': 0.8100, '재현율': 0.8265, 'F1': 0.8182 }
{'오차행렬':
[[56845    19]
 [   17    81]] }
실행 시간: 6.364752531051636
하이퍼파라미터: {'colsample_bytree': 0.6000000000000001, 'gamma': 4.787185438562684, 'learning_rate': 0.005262189759984318, 'max_depth': 5, 'min_child_weight': 6, 'n_estimators': 700, 'reg_alpha': 0.02039

In [14]:
results['xgb_ho_best'] = {'AUC': 0.9721, '정확도': 0.9996, '정밀도': 0.9419, '재현율': 0.8265, 'F1': 0.8804 }
results['xgb_ho_best2'] = {'AUC': 0.9731, '정확도': 0.9994, '정밀도': 0.8100, '재현율': 0.8265, 'F1': 0.8182 }

In [ ]:
# SVC
from sklearn.svm            import LinearSVC

lsvc_model = LinearSVC(random_state=team_rs)

lsvc_search_space = {
    'C': hp.loguniform('C', np.log(0.001), np.log(1000)),  # 정규화 강도 (작을수록 강한 정규화)
    'class_weight': hp.choice('class_weight', [None, 'balanced']),  # 클래스 가중치
    'max_iter': hp.quniform('max_iter', 1000, 10000, 1000),  # 최대 반복 횟수
    'tol': hp.loguniform('tol', np.log(1e-5), np.log(1e-2)),  # 수렴 허용 오차
    'dual': hp.choice('dual', [False, True]),  # dual formulation (n_samples > n_features일 때 False 권장)
}

# 파라미터 지정 - 에러남
# best_params, best_catboost, trials, exec_time = tuner.tune(
#     lsvc_model, X_tr, y_tr, X_val, y_val, lsvc_search_space
# )

In [16]:
import time
import os
from hyperopt import fmin, tpe, STATUS_OK, Trials, hp
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score, confusion_matrix
import numpy as np
from datetime import datetime
from utils.model_utils import save_model

def hyperopt_tune(model_class, search_space, 
                  X_train, y_train, X_test, y_test, 
                  max_evals=100, cv=5, scoring='roc_auc',  random_state=23, is_save_model=True, verbose=True):
    """
    HyperOpt를 사용한 모델 하이퍼파라미터 최적화 함수
    
    Parameters:
    -----------
    model_class : class
        sklearn 모델 클래스 (예: LinearSVC, XGBClassifier)
    search_space : dict
        hyperopt search space 딕셔너리
    X_train : array-like
        학습 데이터
    y_train : array-like
        학습 레이블
    X_test : array-like
        테스트 데이터
    y_test : array-like
        테스트 레이블
    max_evals : int, default=100
        최적화 시도 횟수
    cv : int, default=5
        교차 검증 fold 수
    scoring : str, default='roc_auc'
        평가 지표 ('roc_auc', 'f1', 'recall' 등)
    random_state : int, default=23
        랜덤 시드
    is_save_mode : str or None, default=None
        모델 저장 경로 (None이면 저장 안 함)
    verbose : bool, default=True
        출력 여부 (True: 상세 출력, False: 최소 출력)
    
    Returns:
    --------
    dict : {
        'model': 최적화된 모델,
        'best_params': 최적 하이퍼파라미터,
        'best_score': 최적 점수,
        'trials': hyperopt trials 객체,
        'metrics': 전체 평가 지표 딕셔너리,
        'confusion_matrix': 혼동 행렬,
        'result_dict': 결과 딕셔너리,
        'elapsed_time': 실행 시간
    }
    
    Examples:
    ---------
    >>> from sklearn.svm import LinearSVC
    >>> from hyperopt import hp
    >>> import numpy as np
    >>> 
    >>> # Search Space 정의
    >>> lsvc_search_space = {
    ...     'C': hp.loguniform('C', np.log(0.001), np.log(1000)),
    ...     'class_weight': hp.choice('class_weight', [None, 'balanced']),
    ...     'max_iter': hp.quniform('max_iter', 1000, 10000, 1000),
    ...     'tol': hp.loguniform('tol', np.log(1e-5), np.log(1e-2)),
    ...     'dual': hp.choice('dual', [False, True]),
    ... }
    >>> 
    >>> # HyperOpt 실행
    >>> result = hyperopt_tune(
    ...     model_class=LinearSVC,
    ...     search_space=lsvc_search_space,
    ...     X_train=X_train,
    ...     y_train=y_train,
    ...     X_test=X_test,
    ...     y_test=y_test,
    ...     max_evals=50,
    ...     cv=5,
    ...     scoring='roc_auc',
    ...     random_state=23,
    ...     verbose=True
    ... )
    >>> 
    >>> # XGBoost 예시
    >>> from xgboost import XGBClassifier
    >>> 
    >>> xgb_search_space = {
    ...     'n_estimators': hp.quniform('n_estimators', 100, 1000, 50),
    ...     'max_depth': hp.quniform('max_depth', 3, 10, 1),
    ...     'learning_rate': hp.loguniform('learning_rate', np.log(0.001), np.log(0.3)),
    ...     'subsample': hp.quniform('subsample', 0.5, 1.0, 0.1),
    ...     'colsample_bytree': hp.quniform('colsample_bytree', 0.5, 1.0, 0.1),
    ... }
    >>> 
    >>> result = hyperopt_tune(
    ...     model_class=XGBClassifier,
    ...     search_space=xgb_search_space,
    ...     X_train=X_train,
    ...     y_train=y_train,
    ...     X_test=X_test,
    ...     y_test=y_test,
    ...     max_evals=100,
    ...     is_save_mode=True
    ... )
    >>> 
    >>> # 결과 사용
    >>> best_model = result['model']
    >>> best_params = result['best_params']
    >>> metrics = result['metrics']
    """
    
    # 모델 이름 자동 추출
    model_name = model_class.__name__
    
    if verbose:
        print("=" * 50)
        print(f"  {model_name} 튜닝 시작")
        print("=" * 50)
    
    start_time = time.time()  # 시작 시간 기록
    
    # Objective 함수 정의
    def objective(params):
        # 파라미터 타입 변환 (hyperopt는 float로 반환하므로 int 변환 필요)
        converted_params = {}
        for key, value in params.items():
            # quniform으로 정의된 정수형 파라미터 변환
            if key in ['n_estimators', 'max_depth', 'min_child_weight', 'max_iter', 'scale_pos_weight']:
                converted_params[key] = int(value)
            else:
                converted_params[key] = value
        
        # random_state 추가
        converted_params['random_state'] = random_state
        
        # 모델 생성
        try:
            model = model_class(**converted_params)
        except Exception as e:
            print(f"모델 생성 오류: {e}")
            return {'loss': 1.0, 'status': STATUS_OK}
        
        # 교차 검증
        try:
            scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=scoring)
            score = scores.mean()
        except Exception as e:
            print(f"교차 검증 오류: {e}")
            return {'loss': 1.0, 'status': STATUS_OK}
        
        # HyperOpt는 최소화하므로 음수로 반환
        return {'loss': -score, 'status': STATUS_OK}
    
    # 최적화 실행
    trials = Trials()
    best_params = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        rstate=np.random.default_rng(random_state)
    )
    
    # 걸린 시간 계산
    elapsed_time = time.time() - start_time
    if verbose:
        print(f"튜닝 시간: {elapsed_time:.2f}초")
    
    # 최적 점수 추출
    best_score = -trials.best_trial['result']['loss']
    if verbose:
        print(f"최적 {scoring}: {best_score:.4f}")
    
    # best_params 변환 (choice 타입 처리)
    final_params = {}
    for key, value in best_params.items():
        # choice 파라미터 처리
        if key in search_space:
            space_def = search_space[key]
            # hp.choice인 경우 원래 값으로 변환
            if hasattr(space_def, 'name') and 'choice' in str(type(space_def)):
                # search_space에서 choice 옵션 추출
                choice_options = space_def.pos_args[0].obj
                final_params[key] = choice_options[int(value)]
            # quniform으로 정의된 정수형 파라미터
            elif key in ['n_estimators', 'max_depth', 'min_child_weight', 'max_iter', 'scale_pos_weight']:
                final_params[key] = int(value)
            else:
                final_params[key] = float(value)
        else:
            final_params[key] = value
    
    # random_state 추가
    final_params['random_state'] = random_state
    
    # 최종 모델 학습
    final_model = model_class(**final_params)
    final_model.fit(X_train, y_train)
    
    # 예측
    y_pred = final_model.predict(X_test)
    
    # 확률 예측 (가능한 경우)
    try:
        if hasattr(final_model, 'predict_proba'):
            y_proba = final_model.predict_proba(X_test)[:, 1]
        elif hasattr(final_model, 'decision_function'):
            y_proba = final_model.decision_function(X_test)
        else:
            y_proba = None
    except:
        y_proba = None
    
    # 전체 평가 지표 계산
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred)
    }
    
    # AUC 계산 (확률 예측 가능한 경우)
    if y_proba is not None:
        try:
            metrics['roc_auc'] = roc_auc_score(y_test, y_proba)
        except:
            metrics['roc_auc'] = roc_auc_score(y_test, y_pred)
    
    # 혼동 행렬
    cm = confusion_matrix(y_test, y_pred)
    
    # 결과 출력
    if verbose:
        print("최적 모델의 전체 평가 점수:")
        for metric_name, metric_value in metrics.items():
            print(f"- {metric_name}: {metric_value:.4f}")
        
        print("최적 하이퍼파라미터:")
        for param_name, param_value in final_params.items():
            print(f"-{param_name}: {param_value}")
    
    # 모델 저장
    if is_save_model:
        save_model(final_model, model_name, add_timestamp=True)
        
    
    # 결과 딕셔너리 출력 (사용자가 원하는 형식)        
    
    result_dict = {
        'AUC': round(metrics.get('roc_auc', 0), 4),
        '정확도': round(metrics['accuracy'], 4),
        '정밀도': round(metrics['precision'], 4),
        '재현율': round(metrics['recall'], 4),
        'F1': round(metrics['f1'], 4)
    }
    
    data = {
        "result_dict": result_dict,
        "오차행렬": cm,
        "실행 시간": elapsed_time,
        "하이퍼파라미터": final_params
    }   
     
    if verbose:
        print(data)
    
    # 반환전 result 저장
    import json 
    today = datetime.now().strftime("%Y_%m%d")
    filename = f"{model_name}_{today}.txt"
    save_path = os.path.join("../results/", filename)
        
    # 저장
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    ########################################################################    
    # 불러오기 예시
    # with open(save_path, "r", encoding="utf-8") as f:
    #     loaded_data = json.load(f)
    # print(loaded_data["result_dict"])   # 원하는 부분만 꺼내기
    ########################################################################        
    
    
    # 반환
    return {
        'model': final_model,
        'best_params': final_params,
        'best_score': best_score,
        'trials': trials,
        'metrics': metrics,
        'confusion_matrix': cm,
        'result_dict': result_dict,
        'elapsed_time': elapsed_time
    }
# eof -----------------------------------------------------------------------------    



In [ ]:
from sklearn.svm            import LinearSVC

# lsvc_model = LinearSVC(random_state=team_rs)
lsvc_search_space = {
    'C': hp.loguniform('C', np.log(0.001), np.log(1000)),  # 정규화 강도 (작을수록 강한 정규화)
    'class_weight': hp.choice('class_weight', [None, 'balanced']),  # 클래스 가중치
    'max_iter': hp.quniform('max_iter', 1000, 10000, 1000),  # 최대 반복 횟수
    'tol': hp.loguniform('tol', np.log(1e-5), np.log(1e-2)),  # 수렴 허용 오차
    'dual': hp.choice('dual', [False, True]),  # dual formulation (n_samples > n_features일 때 False 권장)
}

# HyperOpt 실행
result = hyperopt_tune(
    model_class = LinearSVC,
    search_space=lsvc_search_space,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
)
    
# 결과 사용
best_model = result['model']
best_params = result['best_params']
print("\n최종 결과:", result['result_dict'])

NameError: name 'LinearSVC' is not defined

In [ ]:
# 7 시각화
mo.model_metrics_graph(results, 'cb모델 성능지표 비교')

In [ ]:
# BestOpt 찾고 나서 스케일적용 버전 만들어서 모델링하기 
# X_train_scaled, X_test_scaled, scaler = pp.scale_data(X_train, X_test)